# `notebook_3_expeerience`

Альтернативный пайплайн **v2** (баланс кластеров, без Isolation Forest v1).

Рекомендации: `Разработка/Рекомендации_улучшения_Kaggle.md`.  
Выход: `submission2.csv`, артефакты: `artifacts_v2/`.

## Этап 0. Окружение

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from sygnal_clustering.config import DATA_PATH, RANDOM_STATE
from sygnal_clustering.data import load_waveforms
from sygnal_clustering.pipeline_v2 import (
    ARTIFACTS_V2_DIR,
    SUBMISSION2_PATH,
    SygnalClusteringPipelineV2,
    compare_v2_methods,
)

stage0_v2 = {"data_path": str(DATA_PATH), "random_state": RANDOM_STATE}
print(json.dumps(stage0_v2, indent=2))

In [ ]:
from IPython.display import Markdown, display
display(Markdown('''### Этап 0 — ML-архитектор / физик
Запуск v2: критерий отбора — баланс кластеров, не только silhouette v1.'''))

## Этап 1. Сравнение вариантов v2

In [ ]:
X = load_waveforms(DATA_PATH)
comparison_v2 = compare_v2_methods(X, random_state=RANDOM_STATE)
comparison_v2_df = pd.DataFrame(comparison_v2)
display(comparison_v2_df)

In [ ]:
from IPython.display import Markdown, display
display(Markdown('''### Этап 1 — ML-архитектор
Сравнены `balanced_gmm_quantile_psd` и `pc1_psd_tails`. См. `max_cluster_fraction` и доли кластеров в таблице — v2 избегает ~90% в одном классе как v1.

### Этап 1 — физик
GMM на квантиль-нормированном PSD-блоке ближе к двум популяциям + отдельный хвост аномалий.'''))

## Этап 2. Финальная модель v2 и submission2.csv

In [ ]:
pipe_v2 = SygnalClusteringPipelineV2(
    psd_short_len=30,
    anomaly_quantile=0.10,
    use_gmm_primary=True,
    random_state=RANDOM_STATE,
)
labels_v2 = pipe_v2.fit_predict(X)
metrics_v2 = pipe_v2.metrics()
pipe_v2.save_artifacts(ARTIFACTS_V2_DIR)
sub2_path = pipe_v2.save_submission(SUBMISSION2_PATH)
selection_v2 = {**metrics_v2, "submission2": str(sub2_path)}
print(json.dumps(selection_v2, indent=2, ensure_ascii=False))

In [ ]:
from IPython.display import Markdown, display
display(Markdown('''## Итог v2 — выбор модели

### ML-архитектор
Метод **{selection_v2[method]}**. Silhouette **{selection_v2[silhouette]:.4f}**, max доля класса **{selection_v2[max_cluster_fraction]:.3f}**.
Кластеры: **{selection_v2[cluster_0]} / {selection_v2[cluster_1]} / {selection_v2[cluster_2]}**.
Файл: `{selection_v2[submission2]}`. Сравнить accuracy на Kaggle с v1 (`submission.csv`).

### Физик
Более равномерное разбиение γ/нейтронов; класс 2 — хвосты PSD / малый GMM-компонент. Ожидается рост accuracy относительно 0.36568.'''))